# CifarCNN · CIFAR-10 — подбор гиперпараметров

Обучение на полном train-сплите, валидация на test-сплите.  
Архитектура — **точно та же**, что используется в FL-клиентах (`fl_app/models.py`).  
Препроцессинг: только `ToTensor()` — как у FL-клиентов.  

**Как использовать:** меняйте только ячейку «Гиперпараметры», перезапускайте всё сверху вниз.

In [ ]:
# ── Гиперпараметры — редактируйте только этот блок ──────────────────────────

LR           = 0.01   # шаг градиентного спуска
MOMENTUM     = 0.9     # момент SGD
WEIGHT_DECAY = 1e-4    # L2-регуляризация
BATCH_SIZE   = 64      # образцов в батче
EPOCHS       = 60      # число эпох

NUM_WORKERS  = 2       # потоки DataLoader (0 если WSL/Windows)
SEED         = 42
DATA_DIR     = "data"

In [ ]:
import sys
import time

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datasets import load_from_disk
from torch.utils.data import DataLoader

sys.path.insert(0, ".")
from fl_app.models import CifarCNN

torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

In [ ]:
from torchvision.transforms import Compose, RandomCrop, RandomHorizontalFlip, ToTensor

ds = load_from_disk(f"{DATA_DIR}/cifar10")

train_transform = Compose([RandomCrop(32, padding=4), RandomHorizontalFlip(), ToTensor()])
val_transform   = ToTensor()

def apply_train(batch):
    batch["img"] = [train_transform(x) for x in batch["img"]]
    return batch

def apply_val(batch):
    batch["img"] = [val_transform(x) for x in batch["img"]]
    return batch

train_ds = ds["train"].with_transform(apply_train)
test_ds  = ds["test"].with_transform(apply_val)

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)

print(f"Train : {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"Test  : {len(test_ds):,} samples  ({len(test_loader)} batches)")
print("Augmentation: RandomCrop(32, padding=4) + RandomHorizontalFlip (train only)")

In [ ]:
model = CifarCNN(in_channels=3, num_classes=10).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {total_params:,}")
print(model)

In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

print(f"lr={LR}  momentum={MOMENTUM}  weight_decay={WEIGHT_DECAY}  batch_size={BATCH_SIZE}")

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_val_acc = 0.0
best_epoch   = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        x = batch["img"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        run_loss += loss.item()
        correct  += (logits.argmax(1) == y).sum().item()
        total    += y.size(0)
    train_loss = run_loss / len(train_loader)
    train_acc  = correct / total
    scheduler.step()

    # ── Eval ─────────────────────────────────────────────────────────────────
    model.eval()
    run_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in test_loader:
            x = batch["img"].to(device)
            y = batch["label"].to(device)
            logits = model(x)
            run_loss += criterion(logits, y).item()
            correct  += (logits.argmax(1) == y).sum().item()
            total    += y.size(0)
    val_loss = run_loss / len(test_loader)
    val_acc  = correct / total

    current_lr = scheduler.get_last_lr()[0]
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        marker = "  ← best"

    dt = time.perf_counter() - t0
    print(
        f"[{epoch:3d}/{EPOCHS}]  "
        f"train  loss={train_loss:.4f}  acc={train_acc:.4f}  |  "
        f"val  loss={val_loss:.4f}  acc={val_acc:.4f}  |  "
        f"lr={current_lr:.5f}  {dt:.1f}s{marker}"
    )

print(f"\nBest val acc: {best_val_acc:.4f} (epoch {best_epoch})")

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["val_loss"],   label="val")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.4)

ax2.plot(epochs, history["train_acc"], label="train")
ax2.plot(epochs, history["val_acc"],   label="val")
ax2.axhline(best_val_acc, color="gray", linestyle="--", linewidth=0.8,
            label=f"best val {best_val_acc:.4f} (ep {best_epoch})")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.4)

fig.suptitle(
    f"CifarCNN · CIFAR-10   lr={LR}  bs={BATCH_SIZE}  wd={WEIGHT_DECAY}  "
    f"momentum={MOMENTUM}  epochs={EPOCHS}",
    fontsize=11,
)
plt.tight_layout()
plt.show()